# `09 — Graphs: Ways to store graphs + DFS/BFS`

We study:
- **Ways to store graphs**
  - adjacency matrix
  - edge list
  - adjacency list
  - adjacency dict
- **DFS**
  - traversal
  - connected components
  - cycle detection (undirected & directed)
  - topological sort
- **BFS**
  - traversal
  - shortest paths in unweighted graphs (+ path restoration)

Goal:
- Know **O(...)** for each representation/algorithm
- See step-by-step traversal order and structures (stack/queue)
- Be able to restore paths / detect cycles / topo sort


## `1. Graph storage summary`

Let:
- **V = |vertices|**
- **E = |edges|**

| Representation | Memory | "Is edge (u,v)?" | "List neighbors of u" | Best when |
|---|---:|---:|---:|---|
| **Adjacency matrix** | **O(V²)** | O(1) | O(V) | dense graphs |
| **Edge list** | **O(E)** | O(E) | O(E) | algorithms over edges (Kruskal, etc.) |
| **Adjacency list** | **O(V+E)** | O(deg(u)) | **O(deg(u))** | sparse graphs (most common) |
| **Adjacency dict** | **O(V+E)** | ~O(1) avg (dict) | O(deg(u)) | non-integer vertex labels / weighted edges |

In [1]:
from typing import Dict, List, Tuple, Set

V: int = 5
edges: List[Tuple[int, int]] = [(0, 1), (0, 2), (1, 2), (3, 4)]  # directed example from slides  [oai_citation:6‡MSAI.Algo.W07.slides.pdf](sediment://file_00000000cae471f49e20df25ab2a1bfa)

# Adjacency matrix
adj_matrix: List[List[int]] = [[0] * V for _ in range(V)]
for u, v in edges:
    adj_matrix[u][v] = 1

# Edge list (already edges)
edge_list: List[Tuple[int, int]] = edges.copy()

# Adjacency list
adj_list: List[List[int]] = [[] for _ in range(V)]
for u, v in edges:
    adj_list[u].append(v)

# Adjacency dict (list version)
adj_dict: Dict[int, List[int]] = {i: [] for i in range(V)}
for u, v in edges:
    adj_dict[u].append(v)

print("-" * 60)
print("Adjacency matrix:")
for row in adj_matrix:
    print(row)

print("-" * 60)
print("Edge list:", edge_list)

print("-" * 60)
print("Adjacency list:", adj_list)

print("-" * 60)
print("Adjacency dict:", adj_dict)

------------------------------------------------------------
Adjacency matrix:
[0, 1, 1, 0, 0]
[0, 0, 1, 0, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 1]
[0, 0, 0, 0, 0]
------------------------------------------------------------
Edge list: [(0, 1), (0, 2), (1, 2), (3, 4)]
------------------------------------------------------------
Adjacency list: [[1, 2], [2], [], [4], []]
------------------------------------------------------------
Adjacency dict: {0: [1, 2], 1: [2], 2: [], 3: [4], 4: []}


## `2. DFS (Depth First Search) intuition`

DFS explores **depth first**:
- it goes to a neighbor, then neighbor’s neighbor, etc.
- it uses a **stack** (explicit stack or recursion)

Complexity (with adjacency list):
- **O(V + E)** because every vertex is processed once and every edge is inspected once.

In [5]:
from typing import List


def dfs_iter_visual(G: List[List[int]], *, s: int) -> List[int]:
    visited: List[bool] = [False] * len(G)
    stack: List[int] = [s]
    order: List[int] = []

    print("-" * 60)
    print(f"DFS start at {s}")
    print("-" * 60)

    while stack:
        v: int = stack.pop()
        print(f"pop {v:>2} | stack now: {stack}")

        if not visited[v]:
            visited[v] = True
            order.append(v)
            print(f"    visit {v:>2} | order: {order}")

            # push neighbors (reverse to make traversal deterministic/pretty)
            for u in reversed(G[v]):
                if not visited[u]:
                    stack.append(u)
            print(f"    push neighbors of {v}: {G[v]} | stack now: {stack}\n")

    return order


# Example graph (undirected for traversal demo)
G_undirected = [
    [1, 2],     # 0
    [0, 4],     # 1
    [0, 3],     # 2
    [2],        # 3
    [1, 5],     # 4
    [4],        # 5
]
ans = dfs_iter_visual(G_undirected, s=0)

print("-" * 60)
print("DFS order:", ans)
print("-" * 60)

------------------------------------------------------------
DFS start at 0
------------------------------------------------------------
pop  0 | stack now: []
    visit  0 | order: [0]
    push neighbors of 0: [1, 2] | stack now: [2, 1]

pop  1 | stack now: [2]
    visit  1 | order: [0, 1]
    push neighbors of 1: [0, 4] | stack now: [2, 4]

pop  4 | stack now: [2]
    visit  4 | order: [0, 1, 4]
    push neighbors of 4: [1, 5] | stack now: [2, 5]

pop  5 | stack now: [2]
    visit  5 | order: [0, 1, 4, 5]
    push neighbors of 5: [4] | stack now: [2]

pop  2 | stack now: []
    visit  2 | order: [0, 1, 4, 5, 2]
    push neighbors of 2: [0, 3] | stack now: [3]

pop  3 | stack now: []
    visit  3 | order: [0, 1, 4, 5, 2, 3]
    push neighbors of 3: [2] | stack now: []

------------------------------------------------------------
DFS order: [0, 1, 4, 5, 2, 3]
------------------------------------------------------------


## `3. Connected components`

Key observation:
After calling **DFS(v)**, all vertices reachable from v are visited.

For **undirected** graphs, "reachable" means "in the same connected component".
So:
- loop over all vertices
- if vertex is unvisited: run DFS from it → new component.


In [6]:
from typing import List


def connected_components_visual(G: List[List[int]]) -> List[List[int]]:
    n: int = len(G)
    visited: List[bool] = [False] * n
    comps: List[List[int]] = []

    def dfs(v: int, comp: List[int]) -> None:
        visited[v] = True
        comp.append(v)
        for u in G[v]:
            if not visited[u]:
                dfs(u, comp)

    print("-" * 60)
    print("Connected components (undirected)")
    print("-" * 60)

    for v in range(n):
        if not visited[v]:
            comp: List[int] = []
            print(f"\nStart new component from v={v}")
            dfs(v, comp)
            print("Component found:", comp)
            comps.append(comp)

    return comps


# Graph with 2 components
G2 = [
    [1],        # 0
    [0, 2],     # 1
    [1],        # 2
    [4],        # 3
    [3],        # 4
]

components = connected_components_visual(G2)
print("\nAll components:", components)

------------------------------------------------------------
Connected components (undirected)
------------------------------------------------------------

Start new component from v=0
Component found: [0, 1, 2]

Start new component from v=3
Component found: [3, 4]

All components: [[0, 1, 2], [3, 4]]


> **Real-world example**: “Groups of friends” in a social network (undirected edges = friendship).

## `4. Cycle detection in undirected graphs`

While doing DFS(v):
- if you see an already visited neighbor u
- and u is NOT your parent
=> you found a cycle.

Why parent matters:
In undirected graphs, every edge appears twice (v→u and u→v),
so we must ignore the edge back to parent.

In [ ]:
# Find cycle in undirected graph:
from typing import List


def has_cycle_undirected_visual(G: List[List[int]]) -> bool:
    n: int = len(G)
    visited: List[bool] = [False] * n

    def dfs(v: int, p: int) -> bool:
        visited[v] = True
        print(f"Visit v={v}, parent={p}")
        for u in G[v]:
            if not visited[u]:
                print(f"   Tree-edge {v} -> {u}")
                if dfs(u, v):
                    return True
            elif u != p:
                print(f"   Back-edge {v} -> {u} (u visited and u != parent) => CYCLE!")
                return True
        return False

    print("-" * 60)
    print("Cycle detection (undirected)")
    print("-" * 60)

    for v in range(n):
        if not visited[v]:
            print(f"\nStart DFS at {v}")
            if dfs(v, -1):
                return True
    return False


# Graph with a cycle: 0-1-2-0
G_cycle = [
    [1, 2],  # 0
    [0, 2],  # 1
    [0, 1],  # 2
]

ans = has_cycle_undirected_visual(G_cycle)

print("-" * 60)
print("Has cycle?", ans)
print("-" * 60)

------------------------------------------------------------
Cycle detection (undirected)
------------------------------------------------------------

Start DFS at 0
Visit v=0, parent=-1
   Tree-edge 0 -> 1
Visit v=1, parent=0
   Tree-edge 1 -> 2
Visit v=2, parent=1
   Back-edge 2 -> 0 (u visited and u != parent) => CYCLE!
------------------------------------------------------------
Has cycle? True
------------------------------------------------------------


In [12]:
# Directed cycle detection:
from typing import List


def has_cycle_directed_visual(G: List[List[int]]) -> bool:
    n: int = len(G)
    color: List[int] = [0] * n  # 0=white, 1=gray, 2=black

    def dfs(v: int) -> bool:
        color[v] = 1
        print(f"enter v={v} (gray)")
        for u in G[v]:
            if color[u] == 0:
                print(f"  go {v}->{u}")
                if dfs(u):
                    return True
            elif color[u] == 1:
                print(f"  edge {v}->{u} hits gray => CYCLE!")
                return True
        color[v] = 2
        print(f"exit  v={v} (black)")
        return False

    print("-" * 60)
    print("Cycle detection (directed)")
    print("-" * 60)

    for v in range(n):
        if color[v] == 0:
            print(f"\nStart DFS at {v}")
            if dfs(v):
                return True
    return False


# Directed cycle: 0->1->2->0
G_dir_cycle = [
    [1],  # 0
    [2],  # 1
    [0],  # 2
]

ans = has_cycle_directed_visual(G_dir_cycle)

print("-" * 60)
print("Has directed cycle?", ans)
print("-" * 60)

------------------------------------------------------------
Cycle detection (directed)
------------------------------------------------------------

Start DFS at 0
enter v=0 (gray)
  go 0->1
enter v=1 (gray)
  go 1->2
enter v=2 (gray)
  edge 2->0 hits gray => CYCLE!
------------------------------------------------------------
Has directed cycle? True
------------------------------------------------------------


## `5. Topological sort (DAG - Directed Acyclic Graphs)`

 Goal: order vertices so that for every edge u→v, u appears before v.

Topological ordering exists **only if there is no cycle** (graph is a DAG).

DFS trick:
- run DFS
- when you finish processing vertex v (all descendants done), append v to list
- reverse the list at the end

This gives a valid topological order.


In [13]:
from typing import List, Tuple

def topo_sort_visual(G: List[List[int]]) -> List[int]:
    n: int = len(G)
    color: List[int] = [0] * n
    topsort: List[int] = []

    def dfs(v: int) -> None:
        color[v] = 1
        print(f"enter {v}")
        for u in G[v]:
            if color[u] == 0:
                dfs(u)
            elif color[u] == 1:
                raise ValueError("Impossible: graph has a directed cycle")
        topsort.append(v)
        color[v] = 2
        print(f"exit  {v} -> append {v} (current reversed order: {topsort})")

    print("-" * 60)
    print("Topological sort (DFS exit-order)")
    print("-" * 60)

    for v in range(n):
        if color[v] == 0:
            dfs(v)

    topsort.reverse()
    return topsort


# Example DAG: edges define precedence (u must be before v)
G_dag = [
    [1, 2],  # 0 -> 1,2
    [3],     # 1 -> 3
    [3],     # 2 -> 3
    [],      # 3
]

order = topo_sort_visual(G_dag)
print("\nTopological order:", order)

------------------------------------------------------------
Topological sort (DFS exit-order)
------------------------------------------------------------
enter 0
enter 1
enter 3
exit  3 -> append 3 (current reversed order: [3])
exit  1 -> append 1 (current reversed order: [3, 1])
enter 2
exit  2 -> append 2 (current reversed order: [3, 1, 2])
exit  0 -> append 0 (current reversed order: [3, 1, 2, 0])

Topological order: [0, 2, 1, 3]


## `6. BFS (Breadth First Search) intuition`

BFS explores **breadth first**:
- visits all nodes at distance 1, then distance 2, etc.
- uses a **queue** (FIFO)

Complexity:
- **O(V + E)** with adjacency list

In [14]:
from typing import List
from collections import deque


def bfs_visual(G: List[List[int]], *, s: int) -> List[int]:
    visited: List[bool] = [False] * len(G)
    q = deque([s])
    order: List[int] = []

    print("-" * 60)
    print(f"BFS start at {s}")
    print("-" * 60)

    while q:
        v = q.popleft()
        print(f"pop {v:>2} | queue now: {list(q)}")

        if not visited[v]:
            visited[v] = True
            order.append(v)
            print(f"  visit {v:>2} | order: {order}")

            for u in G[v]:
                if not visited[u]:
                    q.append(u)
            print(f"  push neighbors of {v}: {G[v]} | queue now: {list(q)}")

    return order


print("BFS order:", bfs_visual(G_undirected, s=0))

------------------------------------------------------------
BFS start at 0
------------------------------------------------------------
pop  0 | queue now: []
  visit  0 | order: [0]
  push neighbors of 0: [1, 2] | queue now: [1, 2]
pop  1 | queue now: [2]
  visit  1 | order: [0, 1]
  push neighbors of 1: [0, 4] | queue now: [2, 4]
pop  2 | queue now: [4]
  visit  2 | order: [0, 1, 2]
  push neighbors of 2: [0, 3] | queue now: [4, 3]
pop  4 | queue now: [3]
  visit  4 | order: [0, 1, 2, 4]
  push neighbors of 4: [1, 5] | queue now: [3, 5]
pop  3 | queue now: [5]
  visit  3 | order: [0, 1, 2, 4, 3]
  push neighbors of 3: [2] | queue now: [5]
pop  5 | queue now: []
  visit  5 | order: [0, 1, 2, 4, 3, 5]
  push neighbors of 5: [4] | queue now: []
BFS order: [0, 1, 2, 4, 3, 5]


## `7. BFS shortest paths (unweighted graph)`

In the slides we defined:
* $d[i]$ = distance from s to i (number of edges in shortest path)
* $p[i]$ = parent of i in the shortest path tree to reconstruct paths


In [15]:
from collections import deque
from typing import List, Optional

def bfs_shortest_paths_visual(G: List[List[int]], *, s: int) -> Tuple[List[int], List[Optional[int]]]:
    n: int = len(G)
    dist: List[int] = [-1] * n
    parent: List[Optional[int]] = [None] * n

    q = deque([s])
    dist[s] = 0

    print("-" * 60)
    print(f"BFS shortest paths from s={s}")
    print("-" * 60)

    while q:
        v = q.popleft()
        print(f"pop {v}, dist[{v}]={dist[v]} | queue={list(q)}")

        for u in G[v]:
            if dist[u] == -1:
                dist[u] = dist[v] + 1
                parent[u] = v
                q.append(u)
                print(f"  discover u={u}: dist={dist[u]}, parent={v} | queue={list(q)}")

    return dist, parent


def restore_path(parent: List[Optional[int]], *, s: int, t: int) -> List[int]:
    if s == t:
        return [s]
    if parent[t] is None:
        return []
    path: List[int] = []
    cur: Optional[int] = t
    while cur is not None:
        path.append(cur)
        if cur == s:
            break
        cur = parent[cur]
    path.reverse()
    return path


dist, parent = bfs_shortest_paths_visual(G_undirected, s=0)
print("\ndist:", dist)
print("parent:", parent)

target = 5
path = restore_path(parent, s=0, t=target)
print(f"\nShortest path 0 -> {target}:", path, "length =", len(path) - 1)

------------------------------------------------------------
BFS shortest paths from s=0
------------------------------------------------------------
pop 0, dist[0]=0 | queue=[]
  discover u=1: dist=1, parent=0 | queue=[1]
  discover u=2: dist=1, parent=0 | queue=[1, 2]
pop 1, dist[1]=1 | queue=[2]
  discover u=4: dist=2, parent=1 | queue=[2, 4]
pop 2, dist[2]=1 | queue=[4]
  discover u=3: dist=2, parent=2 | queue=[4, 3]
pop 4, dist[4]=2 | queue=[3]
  discover u=5: dist=3, parent=4 | queue=[3, 5]
pop 3, dist[3]=2 | queue=[5]
pop 5, dist[5]=3 | queue=[]

dist: [0, 1, 1, 2, 2, 3]
parent: [None, 0, 0, 2, 1, 4]

Shortest path 0 -> 5: [0, 1, 4, 5] length = 3


## **`Summary`**

### **DFS**
- Uses **stack** (or recursion)
- Best for:
  - connected components (undirected)  
  - cycle detection (undirected parent trick; directed colors 0/1/2)
  - topological sort (DAG) via exit times
- Complexity: **O(V + E)**

### **BFS**
- Uses **queue**
- Gives shortest path lengths in **unweighted** graphs 
- Can restore shortest path via parent array
- Complexity: **O(V + E)**

### **Storage**
- Matrix: O(V²) memory
- Adjacency list/dict: O(V+E) memory